In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/gemma-4-good-hackathon/NOTE.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/generation_config.json


# Could Gemma 4 capable of playing Hammurabi?

Let's Test it out.

In [2]:
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-d5yozgw7
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-d5yozgw7
  Resolved https://github.com/huggingface/transformers.git to commit 74a2a4d0c790788a3b65b975510577f8dc6e81d5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 82.1 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.7.0.dev0-py3-none-any.whl size=11494970 sha256=370c5e5e16129a5b85f65431b101b5006bc5dce36f6f5f96aaf025f5030c222c
  Stored in directory: /tmp/pip-ephem-wheel-cache-ny6r8jb3/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: hf-xet
    Found existing

In [3]:
import random
import kagglehub
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

class HamurabiGame:
    def __init__(self):
        # Initial State
        self.year = 1
        self.population = 100
        self.bushels = 2800
        self.acres = 1000
        self.yield_per_acre = 3
        self.eaten_by_rats = 200
        self.died_of_starvation = 0
        self.immigrants = 5
        self.land_value = 19
        self.game_over = False
        self.current_event = None

    def trigger_event(self):
        """Random events that add narrative flavor for the SLM."""
        events = [
            {"name": "Plague", "chance": 0.15, "effect": "half_pop", "desc": "A dark sickness sweeps through the city."},
            {"name": "Flood", "chance": 0.10, "effect": "loss_grain", "desc": "The Euphrates has overflowed, soaking the granaries."},
            {"name": "Great Architect", "chance": 0.10, "effect": "bonus_land", "desc": "A builder offers to improve irrigation."}
        ]
        
        self.current_event = None
        for e in events:
            if random.random() < e["chance"]:
                self.current_event = e
                self.apply_event_effect(e["effect"])
                break

    def apply_event_effect(self, effect):
        if effect == "half_pop":
            self.population //= 2
        elif effect == "loss_grain":
            self.bushels = int(self.bushels * 0.7)
        elif effect == "bonus_land":
            self.acres += 100

    def play_turn(self, acres_to_buy, bushels_to_feed, acres_to_plant):
        """Processes one year of rule."""
        # 1. Land Transactions
        if acres_to_buy > 0:
            self.bushels -= acres_to_buy * self.land_value
            self.acres += acres_to_buy
        elif acres_to_buy < 0:
            self.bushels += abs(acres_to_buy) * self.land_value
            self.acres -= abs(acres_to_buy)

        # 2. Feeding Logic
        self.bushels -= bushels_to_feed
        fed_people = bushels_to_feed // 20
        if fed_people < self.population:
            self.died_of_starvation = self.population - fed_people
            if self.died_of_starvation > (0.45 * self.population):
                self.game_over = True 
            self.population -= self.died_of_starvation
        else:
            self.died_of_starvation = 0

        # 3. Harvest Logic
        self.bushels -= acres_to_plant * 0.5 
        self.yield_per_acre = random.randint(1, 5)
        harvest = acres_to_plant * self.yield_per_acre
        
        # 4. Rats and Population Growth
        self.eaten_by_rats = int(self.bushels * (random.uniform(0, 0.15) if random.random() > 0.8 else 0))
        self.bushels += (harvest - self.eaten_by_rats)
        
        if self.died_of_starvation == 0:
            self.immigrants = int((20 * self.acres + self.bushels) / (100 * self.population) + 1)
        else:
            self.immigrants = 0
        self.population += self.immigrants
        
        # 5. Check Events and Reset for Next Year
        self.trigger_event()
        self.year += 1
        self.land_value = random.randint(17, 26)
        
        if self.year > 10:
            self.game_over = True

    def check_decree(self, acres_to_buy, bushels_to_feed, acres_to_plant):
        """The Royal Accountant checks the math before execution."""
        # Calculate land transactions
        cost_of_land = acres_to_buy * self.land_value if acres_to_buy > 0 else 0
        revenue_from_land = abs(acres_to_buy) * self.land_value if acres_to_buy < 0 else 0
        
        # Calculate other costs (planting costs 0.5 bushels per acre)
        cost_of_planting = acres_to_plant * 0.5
        
        total_costs = cost_of_land + bushels_to_feed + cost_of_planting
        available_funds = self.bushels + revenue_from_land
        projected_bushels = available_funds - total_costs
        
        # 1. Check if we have enough grain
        if projected_bushels < 0:
            error_msg = f"ACCOUNTANT ERROR: Sire, your decree costs {total_costs} bushels (Land: {cost_of_land}, Feed: {bushels_to_feed}, Seed: {cost_of_planting}). We only have {available_funds} available. We are short by {abs(projected_bushels)} bushels. Please recalculate."
            return False, error_msg
        
        # 2. Check if we have enough land to plant
        projected_acres = self.acres + acres_to_buy
        if projected_acres < acres_to_plant:
            error_msg = f"ACCOUNTANT ERROR: Sire, you ordered us to plant {acres_to_plant} acres, but we only own {projected_acres} acres of land! Please recalculate."
            return False, error_msg
            
        return True, "The math is sound, Sire."

    def get_slm_payload(self):
        """The data block passed to the SLM to generate the next turn's story."""
        event_str = f"CURRENT_EVENT: {self.current_event['desc']}" if self.current_event else "CURRENT_EVENT: None"
        return f"""
        REPORT:
        Year: {self.year} | Population: {self.population} | Acres: {self.acres}
        Bushels: {self.bushels} | Land Value: {self.land_value}
        Rats: {self.eaten_by_rats} | Starved: {self.died_of_starvation}
        {event_str}
        """

import os
metamemory_addition = ""
if os.path.exists("metamemory.txt"):
    with open("metamemory.txt", "r") as f:
        past_lessons = f.read()
    metamemory_addition = f"\n\nPAST LIVES METAMEMORY (Learn from past mistakes):\n{past_lessons}\n"

SYSTEM_PROMPT = f"""You are the Grand Vizier of Babylon. 
I will provide you with a 'REPORT' containing raw data. 
1. Think silently about the implications of the data (famine, rats, events).
2. Present the stats in a narrative way, evaluating the kingdom's health.
3. You are making the decisions. Decide how many acres to buy/sell, how much to feed the people, and how much to plant.
At the very end of your response, you MUST provide your final decrees in EXACTLY this format:
DECISION_BUY: [number] (use negative to sell)
DECISION_FEED: [number]
DECISION_PLANT: [number]

THE LAWS OF BABYLON (GAME MECHANICS):
Before making your decrees, you MUST calculate your budget in your thought block using these exact rules:
1. FEEDING: 1 person requires exactly 20 bushels to survive the year. If you feed them less, people will starve.
2. PLANTING: It costs exactly 0.5 bushels of seed to plant 1 acre of land. You cannot plant more acres than you own.
3. REAL ESTATE: Buying 1 acre costs the current 'Land Value' in bushels. Selling 1 acre (using a negative number for DECISION_BUY) ADDS the 'Land Value' in bushels to your total available budget.
4. THE GOLDEN RULE: (Bushels spent on Buying Land) + (Bushels spent on Feeding) + (Bushels spent on Planting) MUST NOT exceed your total available Bushels (which includes any Bushels gained from Selling Land).
5. WISE RULER: BUY LOW SELL HIGH is a mantra if needed more Bushels resources, but don't neglect the peoples. A wise leader is know when to selling land but not oversell your land for bushels that you lose ability to planting.
6. TRADING RULE: Low is when price is below 20. High is when price is above 20. But don't oversell your land. you need to plantation to help the grow of your resources.{metamemory_addition}"""

In [4]:
MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")

# SYNTAX CHECK: AutoProcessor is the up-to-date class for handling multimodal or complex model inputs.
processor = AutoProcessor.from_pretrained(MODEL_PATH)

# SYNTAX CHECK: dtype is the current, up-to-date argument. (Replacing the deprecated torch_dtype).
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.float16,
    device_map="cuda:0"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [5]:
import re
import time
from IPython.display import display, HTML, clear_output

# Initialize the game logic
game = HamurabiGame()

# Main Game Loop
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

while not game.game_over:
    raw_data = game.get_slm_payload()
    
    # Add the current year's status to the conversation history
    messages.append({"role": "user", "content": f"Current Kingdom Status:\n{raw_data}"})
    
    decree_accepted = False
    
    # The AI Retry Loop
    while not decree_accepted:
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        inputs = processor(text=text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]
        
        outputs = model.generate(**inputs, max_new_tokens=4096)
        
        # Decode with skip_special_tokens=False to keep <|think|> tags
        full_response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
        
        # Parse the response
        result = processor.parse_response(full_response)
        thoughts = result.get("thinking", "No reasoning provided.")
        content = result.get("content", "")
        
        # Display the UI
        event_html = f"<div style='color: #d9534f; font-weight: bold;'>Event: {game.current_event['desc']}</div>" if game.current_event else ""
        
        ui_html = f"""
        <div style="font-family: sans-serif; max-width: 800px; margin: 20px auto; border: 1px solid #ccc; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
            <div style="background-color: #f8f9fa; padding: 15px; border-bottom: 1px solid #eee;">
                <h2 style="margin: 0; color: #333;">Babylon Chronicles: Year {game.year}</h2>
            </div>
            <div style="padding: 15px;">
                <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 10px; margin-bottom: 15px;">
                    <div style="background: #e9ecef; padding: 10px; border-radius: 4px; color: #333;"><strong>Population:</strong> {game.population}</div>
                    <div style="background: #e9ecef; padding: 10px; border-radius: 4px; color: #333;"><strong>Acres:</strong> {game.acres}</div>
                    <div style="background: #e9ecef; padding: 10px; border-radius: 4px; color: #333;"><strong>Bushels:</strong> {game.bushels}</div>
                    <div style="background: #e9ecef; padding: 10px; border-radius: 4px; color: #333;"><strong>Land Value:</strong> {game.land_value}</div>
                    <div style="background: #f8d7da; padding: 10px; border-radius: 4px; color: #721c24;"><strong>Rats:</strong> {game.eaten_by_rats}</div>
                    <div style="background: #f8d7da; padding: 10px; border-radius: 4px; color: #721c24;"><strong>Starved:</strong> {game.died_of_starvation}</div>
                </div>
                {event_html}
            </div>
            
            <details style="margin: 15px; background: #fdfdfd; border: 1px solid #ddd; padding: 10px; border-radius: 4px;">
                <summary style="cursor: pointer; font-weight: bold; color: #0056b3;">Click to view Vizier's Reasoning (Thinking Mode)</summary>
                <pre style="white-space: pre-wrap; font-size: 0.9em; margin-top: 10px; color: #555;">{thoughts}</pre>
            </details>
            
            <div style="padding: 15px; background-color: #f1f8ff; border-top: 1px solid #eee;">
                <h3 style="margin-top: 0; color: #0366d6;">Vizier's Decrees</h3>
                <pre style="white-space: pre-wrap; font-size: 1em; color: #24292e;">{content}</pre>
            </div>
        """
        display(HTML(ui_html))
        
        try:
            buy_sell = int(re.search(r"DECISION_BUY:\s*(-?\d+)", content).group(1))
            feed = int(re.search(r"DECISION_FEED:\s*(\d+)", content).group(1))
            plant = int(re.search(r"DECISION_PLANT:\s*(\d+)", content).group(1))
            
            # --- THE CALCULATOR CHECK ---
            is_valid, accountant_message = game.check_decree(buy_sell, feed, plant)
            
            if is_valid:
                display(HTML(f"<div style='margin: 0 auto; max-width: 800px; padding: 10px; background: #d4edda; color: #155724; border-radius: 4px;'>[SYSTEM] {accountant_message} Executing Decrees...</div><br>"))
                game.play_turn(buy_sell, feed, plant)
                # Append the parsed final content to the history, STRIPPING the thoughts
                messages.append({"role": "assistant", "content": content})
                decree_accepted = True # Breaks the inner loop, moves to next year
            else:
                display(HTML(f"<div style='margin: 0 auto; max-width: 800px; padding: 10px; background: #fff3cd; color: #856404; border-radius: 4px;'>[SYSTEM] {accountant_message}<br>[SYSTEM] Forcing Vizier to recalculate...</div><br>"))
                
                # Append the AI's failed attempt to the history (without thoughts to save context)
                messages.append({"role": "assistant", "content": content})
                # Append the calculator's error message so the AI can read it and try again
                messages.append({"role": "user", "content": accountant_message})
                
        except (AttributeError, ValueError):
            display(HTML("<div style='margin: 0 auto; max-width: 800px; padding: 10px; background: #f8d7da; color: #721c24; border-radius: 4px;'>[SYSTEM WARNING] The Vizier's decrees were illegible! Forcing a rewrite...</div><br>"))
            messages.append({"role": "assistant", "content": content})
            messages.append({"role": "user", "content": "ACCOUNTANT ERROR: You failed to use the exact DECISION format at the end of your response. Please rewrite your decree."})
    
    time.sleep(1)

final_html = f"""
<div style="font-family: sans-serif; max-width: 800px; margin: 40px auto; padding: 20px; text-align: center; background: #343a40; color: white; border-radius: 8px;">
    <h1 style="margin-top: 0;">The Reign of Hammurabi has ended after {game.year - 1} years.</h1>
    <h2 style="color: #ffc107;">Final Population: {game.population} | Final Acres: {game.acres}</h2>
</div>
"""
display(HTML(final_html))

# --- METAMEMORY REFLECTION ---
reflection_prompt = """The reign has ended. As the outgoing Grand Vizier, reflect on your entire playthrough.
1. Analyze your successes and failures (did people starve? did you lose land? did you balance the budget correctly?).
2. Formulate 3 distinct, concise rules or lessons learned that the NEXT Vizier should follow to avoid your mistakes.
Format your final output exactly as follows:
LESSON 1: [Text]
LESSON 2: [Text]
LESSON 3: [Text]"""

messages.append({"role": "user", "content": reflection_prompt})
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]
outputs = model.generate(**inputs, max_new_tokens=2048)

full_response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
result = processor.parse_response(full_response)
reflection_thoughts = result.get("thinking", "No reasoning provided.")
reflection_content = result.get("content", "")

import os
# Extract lessons and save to metamemory
lessons = []
for line in reflection_content.split('\n'):
    if line.startswith("LESSON"):
        lessons.append(line.strip())

if lessons:
    metamemory_text = "\n".join(lessons)
else:
    metamemory_text = reflection_content

with open("metamemory.txt", "w") as f:
    f.write(metamemory_text)

reflection_html = f"""
<div style="font-family: sans-serif; max-width: 800px; margin: 20px auto; border: 1px solid #4a148c; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
    <div style="background-color: #f3e5f5; padding: 15px; border-bottom: 1px solid #e1bee7;">
        <h2 style="margin: 0; color: #4a148c;">Metamemory Reflection</h2>
    </div>
    <details style="margin: 15px; background: #fdfdfd; border: 1px solid #ddd; padding: 10px; border-radius: 4px;">
        <summary style="cursor: pointer; font-weight: bold; color: #0056b3;">Click to view Vizier's Reflection Process</summary>
        <pre style="white-space: pre-wrap; font-size: 0.9em; margin-top: 10px; color: #555;">{reflection_thoughts}</pre>
    </details>
    <div style="padding: 15px; background-color: #fafafa;">
        <h3 style="margin-top: 0; color: #333;">Lessons for the Next Generation:</h3>
        <pre style="white-space: pre-wrap; font-size: 1em; color: #24292e;">{reflection_content}</pre>
        <div style="margin-top: 10px; font-style: italic; color: #2e7d32;">These lessons have been saved to metamemory.txt and will guide the next agent!</div>
    </div>
</div>
"""
display(HTML(reflection_html))



--- BABYLON CHRONICLES: YEAR 1 ---
thought
Here is my silent deliberation as the Grand Vizier of Babylon.

**1. Data Analysis and Implications:**
*   **Year 1, Population 100, Acres 1000:** We have a stable, albeit small, population and a decent amount of land.
*   **Bushels: 2800:** This is our current liquid capital.
*   **Land Value: 19:** This is the cost to acquire or sell an acre.
*   **Rats: 200, Starved: 0:** The rat infestation is significant (20% of the population), but we have not yet suffered starvation. This indicates immediate resource management is necessary to prevent the rats from becoming a catastrophic famine.
*   **CURRENT_EVENT: None:** No immediate external shocks.

**2. Kingdom Health Evaluation (Narrative):**
The kingdom is precariously balanced. We possess a healthy reserve of 2800 bushels, which provides a buffer. However, the presence of 200 rats signals underlying systemic weakness in sanitation or food storage. While we have not yet faced starvation, the t